## 1. setup

In [ ]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

In [ ]:
# Kaggle 上では /kaggle/input、ローカルではリポジトリ直下の data/ を使う
KAGGLE_INPUT = Path("/kaggle/input/competitions/spaceship-titanic")
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
    OUTPUT_DIR = Path("/kaggle/working")
else:
    PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
    DATA_DIR = PROJECT_ROOT / "data"
    OUTPUT_DIR = PROJECT_ROOT / "submissions"
    OUTPUT_DIR.mkdir(exist_ok=True)

ID_COL = "PassengerId"
TARGET = "Transported"

# fold 分割とモデルの seed を揃えて変え、seed 平均で分割の当たり外れをならす
SEEDS = [0, 1, 2]
N_SPLITS = 5
THRESHOLD = 0.5

SPEND_SERVICES = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
LUXURY_SERVICES = ["RoomService", "Spa", "VRDeck"]
BASIC_SERVICES = ["FoodCourt", "ShoppingMall"]
CABINS = ["Deck", "Num", "Side"]
NAMES = ["FirstName", "FamilyName", "Name"]
CATEGORICAL_COLS = ["HomePlanet", "Destination", "VIP", "Deck", "Side", *NAMES]

# oof_acc 0.8159 -> 0.8148 に低下したため不採用
USE_LOGICAL_IMPUTATION = False

LGB_PARAMS = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.1,
    "num_leaves": 16,
    # 上限は大きく取り、実際の木の数は early stopping で決める
    "n_estimators": 100_000,
    "verbose": -1,
}
LGB_EARLY_STOPPING_ROUNDS = 100

CAT_PARAMS = {
    "iterations": 10_000,
    "learning_rate": 0.05,
    "eval_metric": "AUC",
    "early_stopping_rounds": 200,
    "verbose": 0,
}

# OOF accuracy は cat 0.8214、lgb との平均 0.8185 で、cat 単体の方が高い
SUBMIT_MODEL = "cat"

## 2. load

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
print(train.shape, test.shape)

In [ ]:
train.info()

In [ ]:
train.isnull().sum()

## 3. features

In [ ]:
def impute_by_logic(df):
    # コールドスリープ中の人は支出できない → 支出の欠損を0で埋める
    cryo = df["CryoSleep"] == 1
    df.loc[cryo, SPEND_SERVICES] = df.loc[cryo, SPEND_SERVICES].fillna(0)

    # 支出がある人はコールドスリープしていない → CryoSleep の欠損を0で埋める
    has_spend = df[SPEND_SERVICES].sum(axis=1) > 0
    df.loc[df["CryoSleep"].isna() & has_spend, "CryoSleep"] = 0.0
    return df


def split_name(df):
    return df["Name"].str.split(" ", expand=True).set_axis(["FirstName", "FamilyName"], axis=1)


def build_features(df, family_counts):
    df = df.copy()

    # NaN を残したいので astype(bool) ではなく map で数値化する
    df["CryoSleep"] = df["CryoSleep"].map({True: 1.0, False: 0.0})

    df[CABINS] = df["Cabin"].str.split("/", expand=True)
    df["Num"] = df["Num"].astype("float")

    if USE_LOGICAL_IMPUTATION:
        df = impute_by_logic(df)

    df["TotalSpend"] = df[SPEND_SERVICES].sum(axis=1)
    df["NoSpend"] = (df["TotalSpend"] == 0).astype(float)
    df["SpendCount"] = (df[SPEND_SERVICES] > 0).sum(axis=1)
    df["LuxurySpend"] = df[LUXURY_SERVICES].sum(axis=1)
    df["BasicSpend"] = df[BASIC_SERVICES].sum(axis=1)

    df[["FirstName", "FamilyName"]] = split_name(df)
    df["FamilySize"] = df["FamilyName"].map(family_counts).astype("float")

    # train と test で category の水準が揃わなくても、LightGBM は predict 時に学習時の水準へ読み替える
    df[CATEGORICAL_COLS] = df[CATEGORICAL_COLS].astype("category")
    return df


# 同じ家族が train と test に分かれて乗っているため、両方を合わせて数える
family_counts = split_name(pd.concat([train, test]))["FamilyName"].value_counts()

train = build_features(train, family_counts)
test = build_features(test, family_counts)

## 4. dataset

In [ ]:
# ["FamilySize"] を追加: va 0.82 -> 0.81 に低下したため不採用
# ["NoSpend", "SpendCount"] を追加: va 変化なし、LB 0.806 -> 0.805 に低下したため不採用
# "VIP" を除外: va 0.815+-0.004
FEATURES = [
    "Age",
    "TotalSpend",
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP",
    *CABINS,
    *SPEND_SERVICES,
]  # va: 0.816+-0.005

x_train, y_train = train[FEATURES], train[TARGET]
x_test = test[FEATURES]
x_train.info()

# CatBoost はカテゴリ列に NaN を受け付けないため、文字列に変換する（NaN は "nan" になる）
CAT_COLS = x_train.select_dtypes("category").columns.tolist()


def to_catboost_format(x):
    return x.astype(dict.fromkeys(CAT_COLS, str))


x_train_cb = to_catboost_format(x_train)
x_test_cb = to_catboost_format(x_test)

## 5. model train

In [ ]:
def train_lgb(x_tr, y_tr, x_va, y_va, seed):
    model = lgb.LGBMClassifier(**LGB_PARAMS, random_state=seed)
    model.fit(
        x_tr,
        y_tr,
        eval_set=[(x_va, y_va)],
        callbacks=[lgb.early_stopping(LGB_EARLY_STOPPING_ROUNDS, verbose=False)],
    )
    return model


def train_cat(x_tr, y_tr, x_va, y_va, seed):
    model = CatBoostClassifier(**CAT_PARAMS, random_seed=seed)
    model.fit(x_tr, y_tr, cat_features=CAT_COLS, eval_set=(x_va, y_va))
    return model


def run_cv(train_fn, x, y, x_te, seed):
    oof = np.zeros(len(x))
    test_pred = np.zeros(len(x_te))
    folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed).split(x, y)
    for idx_tr, idx_va in folds:
        model = train_fn(x.iloc[idx_tr], y.iloc[idx_tr], x.iloc[idx_va], y.iloc[idx_va], seed)
        oof[idx_va] = model.predict_proba(x.iloc[idx_va])[:, 1]
        test_pred += model.predict_proba(x_te)[:, 1] / N_SPLITS
    return oof, test_pred


def run_seeds(name, train_fn, x, y, x_te):
    oofs, preds = [], []
    for seed in SEEDS:
        oof, pred = run_cv(train_fn, x, y, x_te, seed)
        print(f"{name} seed={seed}: {accuracy_score(y, oof > THRESHOLD):.4f}")
        oofs.append(oof)
        preds.append(pred)
    oof, pred = np.mean(oofs, axis=0), np.mean(preds, axis=0)
    print(f"{name} seed平均: {accuracy_score(y, oof > THRESHOLD):.4f}\n")
    return oof, pred

In [ ]:
results = {
    "lgb": run_seeds("lgb", train_lgb, x_train, y_train, x_test),
    "cat": run_seeds("cat", train_cat, x_train_cb, y_train, x_test_cb),
}
results["ensemble"] = tuple(np.mean([results["lgb"][i], results["cat"][i]], axis=0) for i in range(2))
print(f"ensemble: {accuracy_score(y_train, results['ensemble'][0] > THRESHOLD):.4f}")

## 6. model inference

In [ ]:
y_test_proba = results[SUBMIT_MODEL][1]

df_submit = pd.DataFrame({ID_COL: test[ID_COL], TARGET: y_test_proba > THRESHOLD})
display(df_submit.head(5))
df_submit.to_csv(OUTPUT_DIR / "submission.csv", index=False)